# 🎙️ 会議音声の文字起こし＋話者ラベリング（Colab版）

ブラウザだけで MP3 を文字起こしし、話者（話者A/B/C）と声からの性別ラベルを付けます。

## 使い方（上から順にセルを実行）
1. メニュー「ランタイム → ランタイムのタイプを変更」で **GPU** を選ぶ（任意・高速化）
2. 各セルの ▶️ を上から順に押す
3. HuggingFace の無料トークンを入力（話者分離に必要）
4. 会議 MP3 をアップロード

> ⚠️ **プライバシー**: Colab は Google のクラウドで動くため、音声は一時的に Google のサーバーに送られます（外部の有料APIには送られません）。厳密に外部送信を避けたい場合は手元PCで README の手順を使ってください。

> 🔑 **トークン取得**: (1) https://hf.co/settings/tokens で作成 →(2) https://hf.co/pyannote/speaker-diarization-3.1 で利用規約に同意。どちらも無料。


## 1. ライブラリをインストール（初回のみ・数分）

In [ ]:
!pip -q install faster-whisper pyannote.audio librosa soundfile
print('インストール完了')

## 2. HuggingFace トークンを入力

In [ ]:
from getpass import getpass
HF_TOKEN = getpass('HuggingFace トークンを貼り付けて Enter: ')

## 3. 処理コードを読み込む（実行するだけ）

In [ ]:
import numpy as np, librosa
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline

# 声の高さ(F0)による性別推定のしきい値(Hz)。中間帯は「不明」として誤断定を避ける。
MALE_MAX, FEMALE_MIN = 145.0, 190.0

def estimate_gender(samples, sr):
    if samples.size < sr // 2:
        return '不明'
    f0, vflag, _ = librosa.pyin(samples, fmin=65.0, fmax=400.0, sr=sr)
    v = f0[vflag & ~np.isnan(f0)]
    if v.size == 0:
        return '不明'
    m = float(np.median(v))
    if m <= MALE_MAX:
        return '男性'
    if m >= FEMALE_MIN:
        return '女性'
    return '不明'

def fmt(s):
    mm, ss = divmod(int(s), 60)
    return f'{mm:02d}:{ss:02d}'

def process(path, model_size='small', language='ja', num_speakers=None):
    # 1. 文字起こし
    model = WhisperModel(model_size, device='auto', compute_type='int8')
    raw, _ = model.transcribe(path, language=language, vad_filter=True)
    segs = [(s.start, s.end, s.text.strip()) for s in raw if s.text.strip()]
    # 2. 話者分離
    pipe = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', use_auth_token=HF_TOKEN)
    kw = {'num_speakers': num_speakers} if num_speakers else {}
    dia = pipe(path, **kw)
    turns = [(seg.start, seg.end, spk) for seg, _, spk in dia.itertracks(yield_label=True)]
    # 3. 性別推定用に 16kHz モノラルで読み込み
    samples, sr = librosa.load(path, sr=16000, mono=True)
    speakers = sorted({t[2] for t in turns})
    gender = {}
    for spk in speakers:
        chunks = [samples[int(a*sr):int(b*sr)] for a, b, s in turns if s == spk and b > a]
        gender[spk] = estimate_gender(np.concatenate(chunks), sr) if chunks else '不明'
    labels = {spk: f'話者{chr(65+i)}' for i, spk in enumerate(speakers)}
    # 4. 各セグメントに重なり最大の話者を割り当て
    lines = []
    for st, en, txt in segs:
        best, bo = '?', 0.0
        for a, b, s in turns:
            ov = min(en, b) - max(st, a)
            if ov > bo:
                bo, best = ov, s
        lines.append((st, en, labels.get(best, best), gender.get(best, '不明'), txt))
    return lines

print('準備完了')

## 4. 会議 MP3 をアップロード

In [ ]:
from google.colab import files
uploaded = files.upload()
AUDIO_PATH = next(iter(uploaded))
print('アップロードしたファイル:', AUDIO_PATH)

## 5. 実行

- `model_size`: `tiny`/`base`/`small`/`medium`/`large-v3`。大きいほど高精度・低速。まず `small` を試す。
- `num_speakers`: 話者数が分かれば指定すると分離が安定（None=自動）。


In [ ]:
lines = process(AUDIO_PATH, model_size='small', language='ja', num_speakers=None)

text = '\n'.join(f'[{fmt(st)}] {spk}（{g}）: {txt}' for st, en, spk, g, txt in lines)
print(text)
print(f'\n--- 検出: {len(lines)} 発話 / {len(set(l[2] for l in lines))} 話者 ---')

## 6. 結果をテキストでダウンロード

In [ ]:
with open('transcript.txt', 'w', encoding='utf-8') as f:
    f.write(text + '\n')
from google.colab import files
files.download('transcript.txt')